In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
from mypackages.eRDF import butter_lowpass_filter, DataProcessor
from tqdm import tqdm 
from mypackages.edp_processing import peak_calibration

from matplotlib import rcParams, cycler
from matplotlib.ticker import AutoMinorLocator

from mypackages.plot_style import set_plot_style

set_plot_style()

In [ ]:
#gold peak calibration
from scipy.signal import find_peaks

path = '/home/ABTLUS/victor.secco/data_processing/ED/Au/Au_13_jun_24/Au_inicio/'

#end_name = 'Au_end.csv'
start_name = 'Au_start.csv'

df1 = pd.read_csv(os.path.join(path, start_name), header = None)
#df2 = pd.read_csv(os.path.join(path, end_name), header = None)

data_start = df1[0].values
#data_end = df2[0].values

peaks_start, _ = find_peaks(data_start, distance = 40, height=300)
#peaks_end, _ = find_peaks(data_end, distance = 40, height=500)



calibration = peak_calibration(pixel_positions = peaks_start[3:7])
#calibration_end = peak_calibration(pixel_positions = peaks_end[1:5])

#calibration = (calibration_start +calibration_end)/2

#peaks_end, _ = find_peaks(data_end, distance = 1, height=500)

plt.plot(data_start)
plt.plot(df1[0].values)
plt.scatter(peaks_start[3:7], data_start[peaks_start[3:7]])
plt.show

In [ ]:
#Gr generation
lobato_path = '/home/ABTLUS/victor.secco/data_processing/ED/Lobato_2014.txt'

ds = (calibration)/(2*math.pi) #AuNPs

CdSe =  {1: [34, 1], 2: [48, 1],}
Magnetite = {1: [26, 3], 2: [8, 4],}

path = '/home/ABTLUS/victor.secco/data_processing/ED/CdSe_ePDF/CDSe_Inorg_Lig/'
df1 = pd.read_csv(os.path.join(path, 'CdSe_S2-.csv'), header=None)

raw_data = df1[120].values
start = int(raw_data.shape[0]*0.03)
end =  int(raw_data.shape[0]*0.8)


dp1 = DataProcessor(raw_data, 1,  lobato_path, start, end, ds, CdSe, region = 0.2)
 

_iq = dp1.iq
_sq, _fq = dp1.calculate_SQ_PhiQ(_iq, 0)
_fq_filtered = butter_lowpass_filter(_fq, fs = 50.0, cutoff = 2.0, order = 3)
_r, _Gr = dp1.calculate_Gr(_fq, rmax=80, dr=0.05)

_Gr_Lorch, _ = dp1.low_r_correction(_Gr, 0.025, _r, r_cut = 2)


dp1.plot_results(_sq, _fq, _Gr, _r, _Gr, 0)


In [ ]:
path = '/home/ABTLUS/victor.secco/data_processing/ED/CdSe_ePDF/DATA/iq'

name_list = [x for x in os.listdir(path)]
name_list.sort()

iq_list = []

for name in name_list:
    df = pd.read_csv(os.path.join(path, name))
    iq = df.mean(axis=1).values
    iq_list.append(iq)


calibration = 0.00743649587727647
q = np.linspace(0, len(iq_list[0])*calibration, len(iq))


In [ ]:
fq_list = []

for iq in iq_list:
    start = int(iq.shape[0]*0.03)
    end =  int(iq.shape[0])
    from scipy.interpolate import splrep, splev

    # Perform spline fitting
    # tck contains the spline representation
    tck = splrep(q, iq, s=0)

    # Generate new x values for a smooth curve
    q_spline = np.arange(min(q), max(q), 0.001)

    # Evaluate the spline fit for the new x values
    iq_spline = splev(q_spline, tck)

    #Gr generation
    lobato_path = '/home/ABTLUS/victor.secco/data_processing/ED/Lobato_2014.txt'

    ds = (calibration)/(2*math.pi) #AuNPs

    CdSe =  {1: [34, 1], 2: [48, 1],}

    q_start, q_end = 1.49, 17.843
    indices = np.where((q_spline >= q_start) & (q_spline <= q_end))[0]

    dp1 = DataProcessor(iq, 0.8926770643, lobato_path, start, end, ds, CdSe, region = 0.3)
    

    _iq = dp1.iq
    _sq, _fq = dp1.calculate_SQ_PhiQ(_iq, 0)
    fq_list.append(_fq)

In [ ]:
len(fq_list)

In [ ]:
fig, ax = plt.subplots(1,6, figsize=(15,5))

for i, fq in enumerate(fq_list):

    ax[i].plot(dp1.q, fq, label = name_list[i])
    ax[i].axhline(y=0, color='r', linestyle='dashed')
    ax[i].set_xlim([2,23])    
    ax[-1].set_xticks(np.arange(2, 23, step=5))    
    ax[i].set_yticks([])
    
 
plt.legend()
plt.show()